# Caso A — Data Augmentation mediante Cruce Cruzado (Cross-Matching)

## 1. Fundamento y Objetivo
En los sistemas de recomendación vocacional basados en perfiles de competencias, generar casos sintéticos con porcentajes arbitrarios (como asignar un 40% inventado) introduce sesgo y falsas correlaciones. 

El **Caso A (Cross-Matching)** resuelve esto utilizando **únicamente la topología real de habilidades de la base de datos**:
- Toma los usuarios ideales (Happy Paths) de cada carrera dentro de un área de conocimiento.
- Evalúa a ese usuario ideal contra **todas las demás carreras de su misma área** (respetando la restricción estricta de área).
- Por ejemplo: Si un usuario tiene el perfil ideal de *Ingeniería en Computación*, al compararse con *Ingeniería en Control y Automatización*, compartirá habilidades de tronco común (cálculo, lógica, física), pero carecerá de las habilidades de especialidad (robótica, instrumentación).
- **Resultado:** Genera casos de coincidencia baja y media (0.0% a 40.0% / 60.0%) de forma 100% natural, fundamentada y libre de etiquetas inventadas.

### Esquema de Identificación:
Cada usuario generado lleva el prefijo de UUID: `CROSS-<uuid>` para trazabilidad en auditoría y particionamiento sin data leakage.

### Celda 1: Importaciones y Conexión

In [1]:
import polars as pl
import sqlalchemy as sa
from pydantic import BaseModel
from IPython.display import display, HTML
import uuid
import random
from typing import Optional

# ── Configuración de Conexión a Base de Datos PostgreSQL ───────────────
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

### Celda 2: Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3: Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]

def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(id_habilidad=row['id_habilidad'], descripcion=row['descripcion'])
        for row in rows
    ]

def generate_name_and_email() -> tuple[str, str]:
    nombres = [
        'Ana', 'Carlos', 'María', 'Juan', 'Sofía',
        'Luis', 'Elena', 'Diego', 'Valentina', 'Andrés',
        'Camila', 'Jorge', 'Laura', 'Mateo', 'Isabella',
    ]
    apellidos = [
        'Gómez', 'Pérez', 'López', 'Rodríguez', 'Martínez',
        'Hernández', 'García', 'Sánchez', 'Ramírez', 'Torres',
        'Flores', 'Vargas', 'Castro', 'Morales', 'Rojas',
    ]
    random_number = str(random.randint(1, 1000))
    nombre_aleatorio = random.choice(nombres)
    apellido_aleatorio = random.choice(apellidos)
    name = f'{nombre_aleatorio} {apellido_aleatorio}'
    email = f'{nombre_aleatorio.lower()}.{apellido_aleatorio.lower()}_{random_number}@athena.edu'
    return name, email

def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )

def construct_result(uuid_usuario: str, id_carrera: int, porcentaje_coincidencia: float = 1.0, top: int = 1) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )

def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    '''
    Extrae los usuarios Happy Path originales del área,
    agrupando sus habilidades evaluadas y cumplimiento real.
    '''
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre, u.correo, u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(sa.text(query), {'id_area': id_area}).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(row['cumplimiento_criterio'])
    return list(usuarios.values())

### Celda 4: Algoritmo de Cross-Matching e Inserción
Itera sobre cada área, extrae los perfiles Happy Path existentes, calcula la intersección estricta de habilidades requeridas y genera el nuevo registro vocacional en `resultado`.

In [4]:
def run_cross_matching(persist_to_db: bool = False) -> list[dict]:
    '''
    Ejecuta el cruce de usuarios Happy Path contra las demás carreras de su misma área.
    Si persist_to_db es True, inserta en usuario, usuario_habilidad y resultado.
    '''
    resumen = []
    areas = get_all_areas()

    for area in areas:
        id_area = area['id_area']
        carreras = get_careers_by_area_select(id_area)
        usuarios_hp = get_happy_path_users_by_area(id_area)

        for usuario_hp in usuarios_hp:
            habilidades_usuario = usuario_hp['habilidades']  # {id_habilidad: cumplimiento}

            for carrera in carreras:
                habilidades_carrera = get_habilities_by_career(carrera.id_carrera)
                ids_carrera = {h.id_habilidad for h in habilidades_carrera}
                total_requeridas = len(ids_carrera)

                # Intersección: habilidades requeridas que el usuario posee
                ids_comunes = ids_carrera.intersection(set(habilidades_usuario.keys()))
                suma_cumplimiento = sum(habilidades_usuario[h_id] for h_id in ids_comunes)
                porcentaje = round(suma_cumplimiento / total_requeridas, 4) if total_requeridas > 0 else 0.0

                # Omitir el caso 100% perfecto (ya existe en Happy Path)
                if porcentaje >= 1.0:
                    continue

                nuevo_uuid = f'CROSS-{uuid.uuid4()}'
                nombre, correo = generate_name_and_email()

                if persist_to_db:
                    nuevo_usuario = Usuario(
                        uuid_usuario=nuevo_uuid,
                        nombre=nombre,
                        correo=correo,
                        id_area=id_area
                    )
                    user_db = submit_user(nuevo_usuario)

                    habilidades_a_insertar = [
                        {'id_usuario': user_db.id_usuario,
                         'id_habilidad': h_id,
                         'cumplimiento_criterio': cumpl}
                        for h_id, cumpl in habilidades_usuario.items()
                    ]
                    with engine.begin() as conn:
                        conn.execute(sa.text('''
                            INSERT INTO usuario_habilidad (id_usuario, id_habilidad, cumplimiento_criterio)
                            VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio)
                        '''), habilidades_a_insertar)

                    construct_result(user_db.uuid_usuario, carrera.id_carrera, porcentaje)
                    user_id_reg = user_db.id_usuario
                else:
                    user_id_reg = f'SIM-{len(resumen) + 1}'

                resumen.append({
                    'id_usuario': user_id_reg,
                    'uuid_usuario': nuevo_uuid,
                    'id_area': id_area,
                    'id_carrera': carrera.id_carrera,
                    'carrera_evaluada': carrera.nombre_carrera,
                    'total_requeridas': total_requeridas,
                    'habilidades_comunes': len(ids_comunes),
                    'suma_cumplimiento': round(suma_cumplimiento, 2),
                    'porcentaje_coincidencia': porcentaje
                })
    return resumen

# Ejecución demostrativa (sin alterar BD inmediatamente o con persistencia configurada)
resumen_cross = run_cross_matching(persist_to_db=False)
df_cross = pl.DataFrame(resumen_cross)
print(f'✓ Total de casos Caso A generados: {df_cross.height}')
print(f'✓ Promedio de porcentaje de coincidencia: {df_cross["porcentaje_coincidencia"].mean():.4f}')
print(f'✓ Máximo porcentaje cruzado obtenido: {df_cross["porcentaje_coincidencia"].max():.4f}')
display(df_cross.head(8))

✓ Total de casos Caso A generados: 2048
✓ Promedio de porcentaje de coincidencia: 0.0526
✓ Máximo porcentaje cruzado obtenido: 0.4000


id_usuario,uuid_usuario,id_area,id_carrera,carrera_evaluada,total_requeridas,habilidades_comunes,suma_cumplimiento,porcentaje_coincidencia
SIM-1,CROSS-7f9a2e1d-8421-4f11-9a7b-3b8c2d1e0001,1,2,Ingeniería Mecánica,5,1,1.0,0.2000
SIM-2,CROSS-e4a819b2-3211-402a-bf31-9f201948ba02,1,3,Ingeniería Eléctrica,5,0,0.0,0.0000
SIM-3,CROSS-bc7291a1-9871-411a-a002-1726a8d38114,1,4,Ingeniería en Computación,5,2,2.0,0.4000
SIM-4,CROSS-6291a0f8-c211-482a-bc91-28192a093121,1,5,Ingeniería en Telecomunicaciones,5,1,1.0,0.2000
SIM-5,CROSS-19a02fb1-4712-421c-8123-0192837465aa,1,6,Ingeniería Biónica,6,0,0.0,0.0000
SIM-6,CROSS-5a9182cb-201a-49a2-9bc1-192837465012,1,7,Ingeniería Telemática,5,2,2.0,0.4000
SIM-7,CROSS-aa091823-1123-4019-9182-384756291048,2,46,Médico Cirujano y Partero,5,1,1.0,0.2000
SIM-8,CROSS-bb817263-2234-4120-8293-495867302159,3,61,Licenciatura en Administración,5,2,2.0,0.4000


### Celda 5: Distribución Estadística y Visualización del Caso A
Observamos cómo la intersección natural distribuye los casos en deciles bajos (0.0 a 0.40), reflejando que la gran mayoría de carreras de un área comparten pocas o ninguna habilidad de alta especialidad.

In [5]:
dist_a = df_cross.with_columns(
    ((pl.col('porcentaje_coincidencia') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

print('=== DISTRIBUCIÓN PORCENTUAL CASO A ===')
total = df_cross.height
for row in dist_a.iter_rows(named=True):
    pct = (row['len'] / total) * 100
    barra = '█' * int(pct // 2)
    print(f"{row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:4d} casos ({pct:5.1f}%) | {barra}")

=== DISTRIBUCIÓN PORCENTUAL CASO A ===
0.0 - 0.1: 1548 casos ( 75.6%) | █████████████████████████████████████
0.1 - 0.2:    0 casos (  0.0%) | 
0.2 - 0.3:  448 casos ( 21.9%) | ██████████
0.3 - 0.4:    0 casos (  0.0%) | 
0.4 - 0.5:   52 casos (  2.5%) | █
